In [1]:
import geopandas as gpd
import gcsfs
import google.auth
import pandas as pd

import world_cup_vars as wc_vars

credentials, _ = google.auth.default()

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH

In [17]:
route_gdf = pd.read_parquet(
    f"{GCS_FILE_PATH}fct_daily_schedule_rt_route_direction_summary_world_cup.parquet", 
    filesystem = gcsfs.GCSFileSystem(),
    columns = ["service_date", "schedule_name", "feed_key", "route_id", "route_id_cleaned",
               "route_name", "direction_id", "route_type",
               "shape_id", "shape_array_key", 
               "n_trips", "n_shapes", "num_stop_times", "avg_stops_served"
              ],
    filters = [[("schedule_name", "in", wc_vars.socal_names + wc_vars.bay_area_names)]]
)

In [19]:
route_gdf.route_type.unique()

array(['3', '2', '1', '0'], dtype=object)

In [3]:
route_feeds = (
    route_gdf
    .groupby(["schedule_name"])
    .agg({"feed_key": "nunique"})
    .reset_index()
)

In [4]:
subset_feeds = pd.read_parquet(
    f"{GCS_FILE_PATH}feeds_world_cup.parquet", 
    filesystem=gcsfs.GCSFileSystem()
)

subset_feeds_counts = (
    subset_feeds
    .groupby("gtfs_dataset_name")
    .agg({"feed_key": "nunique"})
    .reset_index()
    .rename(columns = {"gtfs_dataset_name": "schedule_name"})
)

In [5]:
m1 = pd.merge(
    route_feeds, 
    subset_feeds_counts,
    on = "schedule_name",
    how = "outer",
    indicator=True
)

m1._merge.value_counts()

_merge
both          18
right_only     5
left_only      0
Name: count, dtype: int64

In [6]:
m1[m1._merge== "right_only"]

,schedule_name,feed_key_x,feed_key_y,_merge
0,ACE Schedule,NaN,1,right_only
1,Amtrak Schedule,NaN,28,right_only
10,Caltrain Schedule,NaN,1,right_only
11,Capitol Corridor Schedule,NaN,3,right_only
14,Inglewood Schedule,NaN,2,right_only


In [7]:
m1[(m1._merge=="both") & (m1.feed_key_x != m1.feed_key_y)]

,schedule_name,feed_key_x,feed_key_y,_merge
4,Bay Area 511 BART Schedule,1.0,2,both
15,LA DOT Schedule,11.0,12,both
17,LA Metro Events Schedule,1.0,3,both
18,LA Metro Rail Schedule,19.0,20,both


In [8]:
route_gdf.feed_key.nunique() # this is ok, overcount a couple, because filtering is by name

51

In [9]:
subset_feeds.feed_key.nunique()

91

In [10]:
shape_geom = gpd.read_parquet(
    f"{GCS_FILE_PATH}dim_shape_arrays_world_cup.parquet",
    storage_options = {"token": credentials},
    #columns = ["shape_array_key", "geometry"]
)

In [11]:
shape_geom.shape

(7568, 4)

In [12]:
shape_geom.feed_key.nunique()

87

In [13]:
shape_geom.shape_array_key.nunique()

7568

In [14]:
# Ok, this is just a couple of feeds that were missing before
gdf = pd.merge(
    route_gdf,
    shape_geom,
    on = ["feed_key", "shape_id"],
    how = "left",
    indicator=True
)

gdf._merge.value_counts()

_merge
both          19085
left_only         0
right_only        0
Name: count, dtype: int64

In [16]:
gdf.route_type

AttributeError: 'DataFrame' object has no attribute 'route_type'

In [15]:
gdf = pd.merge(
    route_gdf,
    shape_geom,
    on = "shape_array_key",
    how = "left",
    indicator=True
)

gdf._merge.value_counts()

_merge
both          19085
left_only         0
right_only        0
Name: count, dtype: int64

In [ ]:
def merge_routes_with_shape_geom(
    route_df: pd.DataFrame,
):
    shape_geom = gpd.read_parquet(
        f"{GCS_FILE_PATH}dim_shape_arrays_world_cup.parquet",
        storage_options = {"token": credentials},
        columns = ["feed_key", "shape_id", "shape_array_key", "geometry"]
    )
    
    gdf = pd.merge(
        route_df,
        shape_geom,
        on = ["feed_key", "shape_id"],
        how = "left",
        indicator=True
    )

    gdf = gpd.GeoDataFrame(gdf, geometry = "geometry")
    
    return gdf

In [ ]:
route_gdf2 = route_gdf.pipe(merge_routes_with_shape_geom)

In [ ]:
route_gdf.shape, route_gdf2.shape

In [ ]:
route_gdf2._merge.value_counts()

## filter for operators, then filter for routes

filtering for routes is really time-consuming

can we just do geospatial pass, grab all the `routes` where shapes get within 3 miles or 10 miles of stadium?

In [ ]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}daily_schedule_rt_route_direction_summary_combined_dates.parquet",
    filesystem = gcsfs.GCSFileSystem(),
)

FutureWarning - `isin` behavior

/tmp/ipykernel_2595/3934463677.py:1: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.

In [ ]:
sofi_dates = [pd.to_datetime(d) for d in wc_vars.sofi_dates]
levi_dates = [pd.to_datetime(d) for d in wc_vars.levi_dates]

In [ ]:
# isin needs to cast to match dtypes, got 
df_socal = df[(df.service_date.isin(sofi_dates)) & (df.schedule_name.isin(wc_vars.socal_names))]
df_norcal = df[(df.service_date.isin(levi_dates)) & (df.schedule_name.isin(wc_vars.bay_area_names))]

In [ ]:
df = pd.concat([df_socal, df_norcal], axis=0, ignore_index=True)

In [ ]:
la_metro_routes = [
    '22__South Bay Dodger Stadium Express',
    '803__ Metro C Line',
    '807__ Metro K Line'
]

ladot_routes = [
    'CE' # commuter express, so many, need to filter these down more
]

In [ ]:
# filter for routes
df[
    (df.schedule_name == "LA DOT Schedule") & 
    (df.route_name.str.contains("CE"))
    ].route_name.value_counts()